# DTD: Review Captions, Adapt Florence-2, Export New M-TACS Captions

This is a **new captioner experiment**, not another M-TACS ablation or an already trained model.
You currently have no approved caption targets. Start with the review template below; training deliberately
stops until real reviewed targets are provided. Old Florence captions are optional **unverified drafts**,
never automatically copied into the target or approved as ground truth.

The ready-to-run second stage uses supervised **decoder-only LoRA** on the original Microsoft Florence-2-base.
The vision encoder and all base weights remain frozen. Input is image pixels plus `<MORE_DETAILED_CAPTION>`;
supervision is a human-approved 5-8-word English description of visible texture. No category label, filename,
or class hint is placed in the prompt. A target is a reviewed reference description, not the DTD class label.

Only official DTD **train** images can supply references. Those references are split again into fit/holdout;
holdout token NLL selects the adapter. The official DTD val/test images are never used to optimize the captioner.
The frozen adapter later generates captions for all three splits. Do not select captioner settings using test F1.

Run cells in order, complete the review, then continue at the reference-validation cell. The notebook is
self-contained on Kaggle: no project Python files need to be attached. It never calls an external teacher/API
or downloads weights. Captioner training defaults to **5 epochs**, independently of the existing **50-epoch M-TACS** runs.
Lower NLL or more readable captions does not prove better downstream classification; that requires a new controlled run.

In [ ]:
# Review needs no Transformers/PEFT installation and no GPU.
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import json
import shutil
import subprocess
import sys
from pathlib import Path
from importlib.util import find_spec

missing_review_packages = [name for name in ("PIL", "IPython") if find_spec(name) is None]
if missing_review_packages:
    raise RuntimeError("Missing image-review packages: " + ", ".join(missing_review_packages))
print("Offline review stage ready. Tokenizer parallelism is disabled; training/export loaders use workers=0.")

In [ ]:
DTD_ROOT = Path(os.getenv("DTD_ROOT", "/kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd"))
DTD_SPLIT = int(os.getenv("DTD_SPLIT", "1"))
PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_DIR = Path(os.getenv("DTD_CAPTION_FINETUNE_OUTPUT", str(
    PROJECT_DIR / "outputs" / f"florence2_dtd_split{DTD_SPLIT}_caption_lora_v1")))
REVIEW_CSV = OUTPUT_DIR / "review" / "caption_references.csv"
REFERENCE_INPUT_CSV = os.getenv("DTD_CAPTION_REFERENCES", "")  # Optional previously reviewed CSV to import once.
DRAFT_CAPTIONS_CSV = os.getenv("DTD_DRAFT_CAPTIONS_CSV", "")  # Optional OLD captions.csv, for unverified drafts only.
REFERENCE_SAMPLE_PER_CLASS = 4  # 188 images by default; increase before creating the template for wider coverage.
MIN_APPROVED_REFERENCES = 50  # Safety floor, NOT a claim that 50 examples are sufficient for useful adaptation.
REFERENCE_VAL_FRACTION = 0.20
REVIEW_SEED = 42

FLORENCE_MODEL_REF = os.getenv("FLORENCE_MODEL_REF",
    "/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi/Florence-2-base")
OFFLINE_ASSETS_ROOT = os.getenv("DTD_OFFLINE_ASSETS_ROOT", "/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi")
CLIP_MODEL_LOCAL_PATH = os.getenv("CLIP_MODEL_LOCAL_PATH", "/kaggle/input/datasets/nguyenducxu/vit-32/model_clip_vit")
RUN_CLIP_AUDIT = True  # Diagnostic only, never a 0.30 semantic hard gate.
CAPTION_SETTINGS = {
    "seed": 42, "epochs": 5, "batch_size": 2, "accumulation": 4,
    "learning_rate": 5e-5, "weight_decay": 0.01,
    "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
    "max_target_tokens": 48, "max_new_tokens": 48, "export_batch_size": 16,
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Review template:", REVIEW_CSV)
print("New artifacts only:", OUTPUT_DIR)

In [ ]:
"""Review-first supervised Florence caption adaptation; embedded in a Kaggle notebook."""
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import re
import sys
from collections import defaultdict
from pathlib import Path, PurePosixPath

def clean_dtd_caption(text):
    import re
    text = str(text).strip()
    for _ in range(3):
        before = text
        text = re.sub(r"^(?:(?:the|this)\s+)?(?:image|picture|photo(?:graph)?)\s+"
                      r"(?:is|shows|depicts|features|contains)\s+", "", text, flags=re.I)
        text = re.sub(r"^(?:(?:a|an|the)\s+)?close[ -]up(?:\s+view)?\s+(?:of|showing)\s+",
                      "", text, flags=re.I)
        text = re.sub(r"^(?:a|an|the)(?:\s+|$)", "", text, flags=re.I)
        if text == before:
            break
    words = re.findall(r"[A-Za-z0-9]+(?:['\u2019][A-Za-z0-9]+)?", text)
    selected = []
    normalized_count = 0
    for word in words:
        count = len(re.findall(r"[A-Za-z0-9]+", word))
        if normalized_count + count > 8:
            break
        selected.append(word)
        normalized_count += count
    return " ".join(selected)


REFERENCE_COLUMNS = ["review_id", "image_id", "draft_caption_unverified", "target_caption",
                     "caption_source", "review_status", "reviewer_id", "teacher_model", "notes"]
EXPORT_COLUMNS = ["image_id", "split", "raw_caption", "final_caption", "caption_valid",
    "leakage_detected", "lexical_leakage_detected", "semantic_leakage_detected",
    "semantic_leakage_score", "semantic_leakage_class", "retry_count", "vlm_model", "prompt_version",
    "caption_input_policy", "semantic_audit_status", "captioner_fingerprint"]
TASK = "<MORE_DETAILED_CAPTION>"
PROMPT_VERSION = "florence2_dtd_decoder_lora_short_v1"


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,
                                    allow_nan=False).encode()).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def read_csv_rows(path, required=()):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):
            raise ValueError(f"Missing/duplicate CSV header: {path}")
        if not set(required).issubset(reader.fieldnames):
            raise ValueError(f"Missing required CSV columns: {sorted(set(required) - set(reader.fieldnames))}")
        rows = list(reader)
    if any(None in row or any(value is None for value in row.values()) for row in rows):
        raise ValueError(f"Malformed/incomplete CSV row: {path}; original file was not modified")
    return rows


def atomic_csv(path, rows, columns):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)


def normalized_caption(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower().replace("_", " ")).strip()


def load_caption_manifest(root, split, expected_classes=47, per_class=40):
    root = Path(root).expanduser().resolve()
    if not 1 <= int(split) <= 10:
        raise ValueError("DTD_SPLIT must be in 1..10")
    images = (root / "images").resolve()
    names = sorted(path.name for path in images.iterdir() if path.is_dir())
    if len(names) != expected_classes:
        raise ValueError(f"Expected {expected_classes} DTD image folders; found {len(names)} at {images}")
    records, seen = [], set()
    for role in ("train", "val", "test"):
        counts = defaultdict(int)
        for raw in (root / "labels" / f"{role}{split}.txt").read_text(encoding="utf-8-sig").splitlines():
            text = raw.strip().replace("\\", "/")
            if not text:
                continue
            parts = PurePosixPath(text).parts
            if len(parts) != 2 or parts[0] not in names or any(part in {"..", "."} for part in parts):
                raise ValueError(f"Unexpected official DTD path: {text!r}")
            image_id = PurePosixPath(*parts).as_posix()
            path = images.joinpath(*parts).resolve()
            if not path.is_relative_to(images) or not path.is_file():
                raise FileNotFoundError(f"DTD image missing/outside images root: {path}")
            if image_id in seen:
                raise ValueError(f"Duplicate image or overlapping official splits: {image_id}")
            seen.add(image_id)
            counts[parts[0]] += 1
            records.append({"image_id": image_id, "image_path": str(path), "split": role})
        if set(counts) != set(names) or any(count != per_class for count in counts.values()):
            raise ValueError(f"Official {role}{split}.txt must contain {per_class} images/class")
    return records, names


def create_reference_template(records, output, per_class=4, seed=42, draft_csv=""):
    output = Path(output)
    if per_class < 1:
        raise ValueError("Review sample size per class must be positive")
    train = {row["image_id"]: row for row in records if row["split"] == "train"}
    if output.is_file():
        existing = read_csv_rows(output, REFERENCE_COLUMNS)
        if any(row.get("image_id") not in train for row in existing):
            raise ValueError("Existing review template has non-training images; use a separate output directory")
        return output  # Never overwrite manual review work on rerun.
    drafts = {}
    if draft_csv:
        for row in read_csv_rows(draft_csv):
            if row.get("image_id") in train:
                drafts[row["image_id"]] = row.get("raw_caption", "").strip() or row.get("final_caption", "")
    by_class = defaultdict(list)
    for image_id in train:
        by_class[image_id.split("/")[0]].append(image_id)
    rows = []
    for category in sorted(by_class):
        ordered = sorted(by_class[category], key=lambda key: object_hash([seed, key]))
        for image_id in ordered[:per_class]:
            rows.append({"review_id": object_hash(["blind-review", image_id])[:12], "image_id": image_id,
                "draft_caption_unverified": drafts.get(image_id, ""), "target_caption": "",
                "caption_source": "", "review_status": "pending", "reviewer_id": "",
                "teacher_model": "", "notes": ""})
    rows.sort(key=lambda row: object_hash(["review-order", seed, row["review_id"]]))
    atomic_csv(output, rows, REFERENCE_COLUMNS)
    return output


def validate_reference_text(text):
    text = " ".join(str(text).split())
    count = len(normalized_caption(text).split())
    if not 5 <= count <= 8:
        raise ValueError("Approved target_caption must contain 5-8 normalized English words; do not truncate references silently")
    if normalized_caption(clean_dtd_caption(text)) != normalized_caption(text):
        raise ValueError("Remove generic image/close-up openings from the reviewed target caption")
    return text


def save_caption_review(template, review_id, caption, reviewer_id, source="human_corrected", teacher_model=""):
    if not str(reviewer_id).strip():
        raise ValueError("Supply the human reviewer_id; approval is a provenance declaration, not an automated correctness check")
    if source not in {"human", "human_corrected", "teacher"}:
        raise ValueError("caption_source must be human, human_corrected or teacher")
    if source == "teacher" and not teacher_model.strip():
        raise ValueError("Record teacher_model for teacher captions, even after human review")
    target = validate_reference_text(caption)
    rows = read_csv_rows(template)
    matches = [row for row in rows if row.get("review_id") == review_id]
    if len(matches) != 1:
        raise ValueError("review_id must identify exactly one template row")
    matches[0].update(target_caption=target, reviewer_id=reviewer_id.strip(), caption_source=source,
                      review_status="approved", teacher_model=teacher_model.strip())
    atomic_csv(template, rows, REFERENCE_COLUMNS)


def approved_reference_pairs(path, records, minimum=50):
    train = {row["image_id"]: row for row in records if row["split"] == "train"}
    rows = read_csv_rows(path)
    required = {"image_id", "target_caption", "caption_source", "review_status", "reviewer_id"}
    if not rows or not required.issubset(rows[0]):
        raise ValueError("Reference CSV must contain image_id,target_caption,caption_source,review_status,reviewer_id")
    seen, approved = set(), []
    for row in rows:
        image_id = row["image_id"].strip()
        if image_id not in train:
            raise ValueError(f"Only official TRAIN images may be reference examples: {image_id}")
        if image_id in seen:
            raise ValueError(f"Duplicate reference image_id: {image_id}")
        seen.add(image_id)
        if row["review_status"].strip().lower() != "approved":
            continue
        source = row["caption_source"].strip().lower()
        if source not in {"human", "human_corrected", "teacher"} or not row["reviewer_id"].strip():
            raise ValueError(f"Approved reference lacks human review provenance: {image_id}")
        if source == "teacher" and not row.get("teacher_model", "").strip():
            raise ValueError(f"Teacher caption requires teacher_model and human approval: {image_id}")
        approved.append({**train[image_id], "target_caption": validate_reference_text(row["target_caption"]),
            "caption_source": source, "reviewer_id": row["reviewer_id"].strip(),
            "teacher_model": row.get("teacher_model", ""), "image_sha256": file_sha256(train[image_id]["image_path"])})
    if len(approved) < minimum:
        raise ValueError(f"Only {len(approved)} approved caption references; need at least {minimum}. "
                         "Review the template and save captions first. Florence drafts are NOT ground truth; "
                         "no model was trained and no reference answers were invented.")
    return sorted(approved, key=lambda row: row["image_id"])


def split_reference_pairs(pairs, fraction=.20, seed=42):
    if not 0 < fraction < .5:
        raise ValueError("Reference validation fraction must be in (0, 0.5)")
    groups = defaultdict(list)
    for row in pairs:
        if row["split"] != "train":
            raise ValueError("Captioner reference train/holdout must both come from official DTD train")
        groups[row["image_id"].split("/")[0]].append(row)
    fit, holdout = [], []
    for group in groups.values():
        group = sorted(group, key=lambda row: object_hash([seed, row["image_id"]]))
        count = min(max(1, round(len(group) * fraction)), len(group) - 1) if len(group) > 1 else 0
        holdout.extend(group[:count])
        fit.extend(group[count:])
    if len(holdout) < 2 or not fit:
        raise ValueError("Need at least two held-out reference images; approve more training images per class")
    return sorted(fit, key=lambda row: row["image_id"]), sorted(holdout, key=lambda row: row["image_id"])


def checkpoint_asset_identity(model_dir):
    model_dir = Path(model_dir)
    files = sorted(path for path in model_dir.iterdir() if path.is_file() and
                   (path.suffix in {".py", ".json", ".safetensors", ".bin", ".txt"}))
    if not any(path.suffix in {".safetensors", ".bin"} for path in files):
        raise ValueError("Checkpoint has no weight files")
    return {path.name: file_sha256(path) for path in files}


def lock_caption_protocol(output, protocol):
    output = Path(output)
    path = output / "caption_training_protocol.json"
    if path.is_file() and json.loads(path.read_text(encoding="utf-8")) != protocol:
        raise ValueError("Changed caption references, assets, code or hyperparameters: choose a new output directory")
    if not path.is_file() and any((output / name).exists() for name in ("last.pt", "best.pt", "best_adapter")):
        raise ValueError("Existing checkpoint lacks this run's protocol; do not adopt another run's weights")
    atomic_json(path, protocol)
    return object_hash(protocol)


def seed_caption_run(seed):
    import numpy as np
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def validate_caption_payload(payload):
    settings = payload["settings"]
    for name in ("epochs", "batch_size", "accumulation", "lora_rank", "lora_alpha",
                 "max_target_tokens", "max_new_tokens", "export_batch_size"):
        if type(settings[name]) is not int or settings[name] < 1:
            raise ValueError(f"{name} must be a positive integer")
    if not math.isfinite(settings["learning_rate"]) or settings["learning_rate"] <= 0:
        raise ValueError("learning_rate must be finite and positive")
    if not math.isfinite(settings["weight_decay"]) or settings["weight_decay"] < 0:
        raise ValueError("weight_decay must be finite and nonnegative")
    if not 0 <= settings["lora_dropout"] < 1:
        raise ValueError("lora_dropout must be in [0, 1)")
    for key in ("settings", "fit", "holdout"):
        if payload[key] != payload["protocol"][key]:
            raise ValueError(f"Worker {key} does not match the locked training protocol")
    seen = set()
    for row in payload["fit"] + payload["holdout"]:
        if row["split"] != "train" or row["image_id"] in seen:
            raise ValueError("Reference sets must be disjoint and contain official training images only")
        seen.add(row["image_id"])
        validate_reference_text(row["target_caption"])
        if file_sha256(row["image_path"]) != row["image_sha256"]:
            raise ValueError("Reference image changed since review/protocol preparation")
    if not payload["fit"] or len(payload["holdout"]) < 2:
        raise ValueError("Prepare approved reference fit and holdout sets first")


class CaptionPairDataset:
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        from PIL import Image, ImageOps
        row = self.pairs[index]
        with Image.open(row["image_path"]) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
        return image, row["target_caption"]


def caption_collate(items, processor, max_target_tokens=48):
    import torch
    images, targets = zip(*items)
    inputs = processor(text=[TASK] * len(images), images=list(images), padding=True, return_tensors="pt")
    tokens = processor.tokenizer(list(targets), padding=True, truncation=False, return_tensors="pt")
    if tokens["input_ids"].shape[1] > max_target_tokens:
        raise ValueError("Reference token sequence exceeds max_target_tokens; inspect targets rather than truncating them")
    labels = tokens["input_ids"].clone()
    labels[tokens["attention_mask"] == 0] = -100
    if not torch.any(labels != -100):
        raise ValueError("Empty reference-token batch")
    return {key: value for key, value in inputs.items() if torch.is_tensor(value)}, labels


def decoder_lora_targets(model):
    import torch
    targets = [name for name, module in model.named_modules() if name.startswith("language_model.")
               and ".decoder." in name and name.rsplit(".", 1)[-1] in {"q_proj", "v_proj"}
               and isinstance(module, torch.nn.Linear)]
    if not targets:
        raise ValueError("No Florence decoder q_proj/v_proj modules found; do not silently adapt the vision encoder")
    return sorted(targets)


def load_caption_model(payload, device, trainable=False):
    import torch
    from transformers import AutoModelForCausalLM, AutoProcessor
    from peft import LoraConfig, PeftModel, get_peft_model
    processor = AutoProcessor.from_pretrained(payload["model"], trust_remote_code=True, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(payload["model"], trust_remote_code=True,
        local_files_only=True, torch_dtype=torch.float32, low_cpu_mem_usage=True, attn_implementation="eager")
    if trainable:
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        targets = decoder_lora_targets(model)
        settings = payload["settings"]
        # Generic PEFT preserves Florence's image-aware forward/generate signature.
        model = get_peft_model(model, LoraConfig(r=settings["lora_rank"], lora_alpha=settings["lora_alpha"],
            lora_dropout=settings["lora_dropout"], target_modules=targets, bias="none"))
        names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
        if not names or any("lora_" not in name or ".decoder." not in name for name in names):
            raise ValueError("Unexpected trainable parameters; vision encoder/base model must remain frozen")
        atomic_json(Path(payload["output"]) / "trainable_parameters.json", {
            "targets": targets, "parameters": names,
            "trainable_count": sum(p.numel() for p in model.parameters() if p.requires_grad),
            "total_count": sum(p.numel() for p in model.parameters())})
    else:
        model = PeftModel.from_pretrained(model, str(Path(payload["output"]) / "best_adapter"),
                                         is_trainable=False, local_files_only=True)
    return model.to(device), processor


def freeze_caption_vision(model):
    base = model.get_base_model()
    if hasattr(base, "vision_tower"):
        base.vision_tower.eval()


def adapter_state(model):
    from peft import get_peft_model_state_dict
    return {name: tensor.detach().cpu().clone() for name, tensor in get_peft_model_state_dict(model).items()}


def restore_adapter(model, state):
    from peft import set_peft_model_state_dict
    set_peft_model_state_dict(model, state)


def atomic_torch_file(path, state):
    import torch
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(state, temporary)
    temporary.replace(path)


def capture_caption_rng(generator):
    import numpy as np
    import torch
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "loader": generator.get_state()}


def restore_caption_rng(state, generator):
    import numpy as np
    import torch
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"]:
        torch.cuda.set_rng_state_all(state["cuda"])
    generator.set_state(state["loader"])


def move_caption_batch(batch, device):
    inputs, labels = batch
    return {key: value.to(device) for key, value in inputs.items()}, labels.to(device)


def caption_autocast(device):
    import torch
    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda")


def evaluate_caption_nll(model, loader, device):
    import torch
    model.eval()
    total, tokens = 0., 0
    with torch.inference_mode():
        for batch in loader:
            inputs, labels = move_caption_batch(batch, device)
            count = int((labels != -100).sum())
            with caption_autocast(device):
                loss = model(**inputs, labels=labels, use_cache=False).loss
            if not torch.isfinite(loss):
                raise ValueError("Non-finite caption validation NLL")
            total += float(loss) * count
            tokens += count
    return total / max(tokens, 1)


def train_caption_epoch(model, loader, optimizer, scheduler, scaler, device, accumulation):
    import torch
    model.train()
    freeze_caption_vision(model)
    optimizer.zero_grad(set_to_none=True)
    total_loss = total_tokens = group_tokens = 0
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    for index, batch in enumerate(loader):
        inputs, labels = move_caption_batch(batch, device)
        count = int((labels != -100).sum())
        with caption_autocast(device):
            loss = model(**inputs, labels=labels, use_cache=False).loss
        if not torch.isfinite(loss):
            raise ValueError("Non-finite caption training loss; last completed epoch is preserved")
        scaler.scale(loss * count).backward()
        total_loss += float(loss.detach()) * count
        total_tokens += count
        group_tokens += count
        if (index + 1) % accumulation == 0 or index + 1 == len(loader):
            scaler.unscale_(optimizer)
            for parameter in parameters:
                if parameter.grad is not None:
                    parameter.grad.div_(group_tokens)
            torch.nn.utils.clip_grad_norm_(parameters, 1.)
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= previous_scale:
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            group_tokens = 0
    return total_loss / max(total_tokens, 1)


def train_captioner(payload, device):
    import torch
    from functools import partial
    from torch.utils.data import DataLoader
    validate_caption_payload(payload)
    settings = payload["settings"]
    output = Path(payload["output"])
    seed_caption_run(settings["seed"])
    fingerprint = lock_caption_protocol(output, payload["protocol"])
    model, processor = load_caption_model(payload, device, trainable=True)
    generator = torch.Generator().manual_seed(settings["seed"])
    collate = partial(caption_collate, processor=processor, max_target_tokens=settings["max_target_tokens"])
    train_loader = DataLoader(CaptionPairDataset(payload["fit"]), batch_size=settings["batch_size"],
        shuffle=True, generator=generator, num_workers=0, collate_fn=collate)
    val_loader = DataLoader(CaptionPairDataset(payload["holdout"]), batch_size=settings["batch_size"],
        shuffle=False, num_workers=0, collate_fn=collate)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=settings["learning_rate"], weight_decay=settings["weight_decay"])
    updates_per_epoch = math.ceil(len(train_loader) / settings["accumulation"])
    total_updates = updates_per_epoch * settings["epochs"]
    warmup = max(1, int(total_updates * .1))
    def factor(step):
        if step < warmup:
            return (step + 1) / warmup
        return max(.01, .5 * (1 + math.cos(math.pi * (step - warmup) / max(total_updates - warmup, 1))))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, factor)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())
    history, start, best_nll, best_epoch = [], 1, float("inf"), 0
    resume_path, best_path = output / "last.pt", output / "best.pt"
    if resume_path.is_file():
        state = torch.load(resume_path, map_location="cpu", weights_only=False)
        if state["identity"] != fingerprint:
            raise ValueError("Resume checkpoint belongs to another caption experiment")
        restore_adapter(model, state["adapter"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        history, best_nll, best_epoch = state["history"], state["best_nll"], state["best_epoch"]
        start = state["epoch"] + 1
        restore_caption_rng(state["rng"], generator)
        print("Resuming captioner at epoch", start, flush=True)
    # DataLoader iteration consumes RNG even without shuffling; preserve exact epoch-boundary resume.
    baseline_rng = capture_caption_rng(generator)
    with model.disable_adapter():
        baseline_nll = evaluate_caption_nll(model, val_loader, device)
    restore_caption_rng(baseline_rng, generator)
    for epoch in range(start, settings["epochs"] + 1):
        training = train_caption_epoch(model, train_loader, optimizer, scheduler, scaler, device, settings["accumulation"])
        validation = evaluate_caption_nll(model, val_loader, device)
        history.append({"epoch": epoch, "train_token_nll": training, "holdout_token_nll": validation})
        if validation < best_nll:
            best_nll, best_epoch = validation, epoch
            atomic_torch_file(best_path, {"identity": fingerprint, "epoch": epoch, "holdout_nll": validation,
                                         "adapter": adapter_state(model)})
        atomic_torch_file(resume_path, {"identity": fingerprint, "epoch": epoch, "adapter": adapter_state(model),
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
            "history": history, "best_nll": best_nll, "best_epoch": best_epoch, "rng": capture_caption_rng(generator)})
        atomic_csv(output / "history.csv", history, ["epoch", "train_token_nll", "holdout_token_nll"])
        print(f"Caption epoch {epoch}/{settings['epochs']}: train_nll={training:.4f}; holdout_nll={validation:.4f}", flush=True)
    best = torch.load(best_path, map_location="cpu", weights_only=False)
    if best["identity"] != fingerprint:
        raise ValueError("Best adapter has the wrong identity")
    restore_adapter(model, best["adapter"])
    adapter_dir = output / "best_adapter"
    model.save_pretrained(adapter_dir, safe_serialization=True, save_embedding_layers=False)
    processor.save_pretrained(output / "processor")
    comparisons = []
    for row in payload["holdout"]:
        with model.disable_adapter():
            before = generate_caption_texts(model, processor, [row], device, settings["max_new_tokens"])[0]
        after = generate_caption_texts(model, processor, [row], device, settings["max_new_tokens"])[0]
        comparisons.append({"image_id": row["image_id"], "reference": row["target_caption"],
            "base_raw_caption": before, "adapted_raw_caption": after,
            "base_short_caption": clean_dtd_caption(before), "adapted_short_caption": clean_dtd_caption(after),
            "human_preference": "", "review_notes": ""})
    comparison_path = output / "holdout_before_after.csv"
    if comparison_path.is_file():
        previous = {row["image_id"]: row for row in read_csv_rows(comparison_path)}
        for row in comparisons:
            old = previous.get(row["image_id"], {})
            if any(old.get(key) != value for key, value in row.items()
                   if key not in {"human_preference", "review_notes"}):
                raise ValueError("Existing holdout comparison differs; preserve manual reviews and use a new output directory")
        # Keep the original file, including human preferences, notes and any extra review columns.
    else:
        atomic_csv(comparison_path, comparisons, list(comparisons[0]))
    summary = {"identity": fingerprint, "epochs_completed": len(history), "best_epoch": best_epoch,
        "baseline_holdout_token_nll": baseline_nll, "best_holdout_token_nll": best_nll,
        "fit_images": len(payload["fit"]), "holdout_images": len(payload["holdout"]),
        "reference_origin": "human_reviewed_train_only", "downstream_f1_improvement_verified": False}
    atomic_json(output / "training_complete.json", summary)
    print(json.dumps(summary, indent=2), flush=True)
    return summary


def generate_caption_texts(model, processor, rows, device, max_new_tokens):
    import torch
    from PIL import Image, ImageOps
    images = []
    for row in rows:
        with Image.open(row["image_path"]) as source:
            images.append(ImageOps.exif_transpose(source).convert("RGB"))
    inputs = processor(text=[TASK] * len(images), images=images, padding=True, return_tensors="pt")
    moved = {key: value.to(device) for key, value in inputs.items() if torch.is_tensor(value)}
    model.eval()
    with torch.inference_mode(), caption_autocast(device):
        generated = model.generate(**moved, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False, use_cache=True)
    decoded = processor.batch_decode(generated, skip_special_tokens=False)
    if len(decoded) != len(rows):
        raise ValueError("Florence returned the wrong number of captions")
    results = []
    for text, image in zip(decoded, images):
        parsed = processor.post_process_generation(text, task=TASK, image_size=(image.width, image.height))
        results.append(str(parsed.get(TASK, text) if isinstance(parsed, dict) else text).strip())
    return results


class CaptionSemanticAuditor:
    def __init__(self, path, names, device):
        import torch
        from transformers import CLIPTextModelWithProjection, CLIPTokenizer
        if not Path(path).is_dir():
            raise FileNotFoundError("Attach local CLIP ViT-B/32 for diagnostic audit, or explicitly disable run_clip_audit")
        self.tokenizer = CLIPTokenizer.from_pretrained(path, local_files_only=True)
        self.model = CLIPTextModelWithProjection.from_pretrained(path, local_files_only=True).to(device).eval()
        self.names, self.device = names, device
        prompts = [template.format(name=name) for name in names
                   for template in ("a photo of {name}", "a scene showing {name}")]
        vectors = self.encode(prompts).reshape(len(names), 2, -1).mean(1)
        self.vectors = torch.nn.functional.normalize(vectors, dim=1)

    def encode(self, texts):
        import torch
        vectors = []
        with torch.inference_mode():
            for start in range(0, len(texts), 128):
                tokens = self.tokenizer(texts[start:start + 128], return_tensors="pt", padding=True,
                    truncation=True, max_length=self.model.config.max_position_embeddings).to(self.device)
                vectors.append(torch.nn.functional.normalize(self.model(**tokens).text_embeds.float(), dim=1))
        return torch.cat(vectors)

    def scores(self, texts):
        scores, indices = (self.encode(texts) @ self.vectors.T).max(1)
        return [(float(score), self.names[int(index)]) for score, index in zip(scores.cpu(), indices.cpu())]


def make_export_row(record, raw, identity, names, semantic=None):
    final = clean_dtd_caption(raw)
    normalized = " " + normalized_caption(final) + " "
    lexical = any(" " + normalized_caption(name) + " " in normalized for name in names)
    valid = 5 <= len(normalized.split()) <= 8 and not lexical
    score, matched = semantic if semantic is not None else ("", "")
    return {"image_id": record["image_id"], "split": record["split"], "raw_caption": raw,
        "final_caption": final, "caption_valid": valid, "leakage_detected": lexical,
        "lexical_leakage_detected": lexical, "semantic_leakage_detected": bool(semantic is not None and score >= .30),
        "semantic_leakage_score": score, "semantic_leakage_class": matched, "retry_count": 0,
        "vlm_model": "microsoft/Florence-2-base+dtd-decoder-lora:" + identity[:16],
        "prompt_version": PROMPT_VERSION, "caption_input_policy": "dtd_lexical_v2",
        "semantic_audit_status": "diagnostic_only" if semantic is not None else "not_run",
        "captioner_fingerprint": identity}


def export_caption_csv(payload, device):
    import torch
    output = Path(payload["output"])
    summary = json.loads((output / "training_complete.json").read_text(encoding="utf-8"))
    training_id = object_hash(payload["protocol"])
    if summary["identity"] != training_id or summary["epochs_completed"] != payload["settings"]["epochs"]:
        raise ValueError("Finish this caption training protocol before exporting all DTD captions")
    records = payload["export_records"]
    if len({row["image_id"] for row in records}) != len(records):
        raise ValueError("Duplicate export image IDs")
    print("Hashing export images to verify dataset content across resumed sessions...", flush=True)
    inventory = [{"image_id": row["image_id"], "split": row["split"],
                  "sha256": file_sha256(row["image_path"])} for row in records]
    manifest = {"training": training_id, "best_adapter": checkpoint_asset_identity(output / "best_adapter"),
        "images": inventory, "prompt": TASK, "version": PROMPT_VERSION, "policy": "dtd_lexical_v2",
        "max_new_tokens": payload["settings"]["max_new_tokens"], "sampling": "greedy",
        "audit_class_names": list(payload["class_names"]),
        "clip": checkpoint_asset_identity(payload["clip"]) if payload["run_clip_audit"] else None,
        "clip_threshold": .30, "semantic_hard_gate": False}
    identity = object_hash(manifest)
    folder, csv_path = output / "exports", output / "exports" / "captions.csv"
    manifest_path = folder / "caption_generation_manifest.json"
    if manifest_path.is_file() and json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise ValueError("Changed caption generation assets/options; use a new export directory, never mix CSV rows")
    if csv_path.exists() and not manifest_path.exists():
        raise ValueError("Existing captions.csv has no matching generation manifest; original file was not modified")
    atomic_json(manifest_path, manifest)
    existing = read_csv_rows(csv_path, EXPORT_COLUMNS) if csv_path.is_file() and csv_path.stat().st_size else []
    expected = {row["image_id"]: row["split"] for row in records}
    seen = set()
    for row in existing:
        if not set(EXPORT_COLUMNS).issubset(row) or row["image_id"] in seen or row["image_id"] not in expected:
            raise ValueError("Invalid schema, duplicate or unexpected image in export CSV")
        if row["captioner_fingerprint"] != identity or row["split"] != expected[row["image_id"]]:
            raise ValueError("Export CSV belongs to another captioner/split")
        seen.add(row["image_id"])
    todo = [row for row in records if row["image_id"] not in seen]
    print(f"Improved captions: cached={len(seen)}, remaining={len(todo)}, output={csv_path}", flush=True)
    if todo:
        model, processor = load_caption_model(payload, device, trainable=False)
        auditor = CaptionSemanticAuditor(payload["clip"], payload["class_names"], device) if payload["run_clip_audit"] else None
        size, offset = payload["settings"]["export_batch_size"], 0
        while offset < len(todo):
            group = todo[offset:offset + size]
            try:
                raw = generate_caption_texts(model, processor, group, device, payload["settings"]["max_new_tokens"])
                scores = auditor.scores([clean_dtd_caption(text) for text in raw]) if auditor else [None] * len(group)
            except torch.cuda.OutOfMemoryError:
                if size == 1:
                    raise
                size = max(1, size // 2)
                torch.cuda.empty_cache()
                print("Export batch reduced to", size, flush=True)
                continue
            if len(raw) != len(group) or len(scores) != len(group):
                raise ValueError("Caption/audit batch count mismatch; no rows from this batch were appended")
            rows = [make_export_row(row, text, identity, payload["class_names"], score)
                    for row, text, score in zip(group, raw, scores)]
            header = not csv_path.exists() or csv_path.stat().st_size == 0
            with csv_path.open("a", encoding="utf-8", newline="") as handle:
                writer = csv.DictWriter(handle, fieldnames=EXPORT_COLUMNS)
                if header:
                    writer.writeheader()
                writer.writerows(rows)
                handle.flush()
                os.fsync(handle.fileno())
            offset += len(group)
            print(f"Exported {len(seen) + offset}/{len(records)}", flush=True)
    final = read_csv_rows(csv_path)
    if {row["image_id"] for row in final} != set(expected):
        raise ValueError("Export is incomplete; rerun export to resume missing images")
    atomic_json(folder / "export_summary.json", {"rows": len(final), "csv_sha256": file_sha256(csv_path),
        "captioner_fingerprint": identity, "policy": "dtd_lexical_v2",
        "valid_captions": sum(row["caption_valid"] == "True" for row in final),
        "human_ground_truth": False, "downstream_improvement_verified": False})
    return csv_path


def caption_worker_main(payload_path, action):
    import torch
    import transformers
    import peft
    if not transformers.__version__.startswith("4.49.") or peft.__version__ != "0.14.0":
        raise RuntimeError("This legacy Florence training notebook uses Transformers 4.49.x and PEFT 0.14.0 in its isolated worker")
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU for real Florence training/export; CPU mocks are only for tests")
    payload = json.loads(Path(payload_path).read_text(encoding="utf-8"))
    if checkpoint_asset_identity(payload["model"]) != payload["protocol"]["base_assets"]:
        raise ValueError("Base Florence assets changed after protocol preparation; do not reuse this run")
    device = torch.device("cuda")
    if action == "train":
        train_captioner(payload, device)
    elif action == "export":
        export_caption_csv(payload, device)
    else:
        raise ValueError("Worker action must be train or export")

## 1. Create the Review Template

The default is 4 train images per class, 188 in total. Sampling uses classes only to cover the dataset;
the blind image viewer below does not show the class or filename. `target_caption` starts empty.
Existing template edits are preserved when this cell is rerun. Keep the output folder when leaving Kaggle.

Set `DTD_DRAFT_CAPTIONS_CSV` only to show an old generated caption as an unverified suggestion.
You can leave it empty and describe images directly. A teacher may suggest text from pixels alone,
but record its model/version and have a human inspect each accepted answer. No teacher is called here.

In [ ]:
caption_records, dtd_class_names = load_caption_manifest(DTD_ROOT, DTD_SPLIT)
if REFERENCE_INPUT_CSV and not REVIEW_CSV.exists():
    source = Path(REFERENCE_INPUT_CSV)
    if not source.is_file():
        raise FileNotFoundError(f"Reviewed reference CSV not found: {source}")
    REVIEW_CSV.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, REVIEW_CSV)
    atomic_json(REVIEW_CSV.parent / "import_source.json", {"source": str(source), "sha256": file_sha256(source)})
create_reference_template(caption_records, REVIEW_CSV, REFERENCE_SAMPLE_PER_CLASS, REVIEW_SEED, DRAFT_CAPTIONS_CSV)
review_rows = read_csv_rows(REVIEW_CSV)
for role in ("train", "val", "test"):
    print(role, sum(row["split"] == role for row in caption_records))
print("Review rows:", len(review_rows), "| approved:", sum(row.get("review_status") == "approved" for row in review_rows))
print("Review CSV:", REVIEW_CSV)

review_guide = """CAPTION REVIEW PROTOCOL v1
- Inspect only the displayed training-image pixels; do not infer a target from its category/filename.
- Write 5-8 English words describing visible material, color, local shapes, repetition or surface structure.
- Avoid boilerplate ('The image is', 'a close-up of'), unsupported objects and invisible properties.
- Describe what is visible, not a guess at the DTD class. Do not look up a label to construct the target.
- Targets need not match a single exact wording. Consistency and visual truth matter more than verbosity.
- Reject uncertain drafts: keep review_status=pending and explain uncertainty in notes.
- Approval requires target_caption, caption_source (human/human_corrected/teacher), reviewer_id,
  review_status=approved, and teacher_model when a teacher proposed the text.
- A teacher proposal is not ground truth until a human has inspected and accepted it.
- Direct class-word overlap is reported later; reference correctness and leakage screening are distinct.
- Do not use validation/test images or test classification scores to revise reference captions.
"""
(REVIEW_CSV.parent / "REVIEW_GUIDE.txt").write_text(review_guide, encoding="utf-8")
teacher_prompt = """Optional image-only teacher prompt (not executed by this notebook):
Describe only the visible texture in this image in 5-8 English words. Focus on material, color,
local shapes, spacing, repetition or surface structure. Do not name a texture category, infer
a dataset label, mention filenames or use generic image/close-up boilerplate. Do not invent details.
If the pixels do not support a confident description, return REVIEW_NEEDED instead.
Human review is required before this text may become a training target.
"""
(REVIEW_CSV.parent / "OPTIONAL_TEACHER_PROMPT.txt").write_text(teacher_prompt, encoding="utf-8")
print(review_guide)

In [ ]:
from IPython.display import display
from PIL import Image, ImageOps

def show_caption_review(index=0):
    rows = read_csv_rows(REVIEW_CSV)
    if not 0 <= index < len(rows):
        raise IndexError(f"Review index must be in 0..{len(rows)-1}")
    row = rows[index]
    record = next(record for record in caption_records if record["image_id"] == row["image_id"])
    if record["split"] != "train":
        raise ValueError("The review viewer accepts official training images only")
    with Image.open(record["image_path"]) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
    image.thumbnail((700, 500))
    display(image)
    print("Review index:", index, "| blind review_id:", row["review_id"], "| status:", row["review_status"])
    print("UNVERIFIED draft (optional):", row["draft_caption_unverified"] or "[none]")
    print("Your target:", row["target_caption"] or "[empty]")
    print("Reviewer:", row["reviewer_id"] or "[not reviewed]")
    return row["review_id"]

current_review_id = show_caption_review(0)

Inspect another image with `current_review_id = show_caption_review(1)` (change the index).
After looking at the image, fill and run the following cell to save that review. The default does nothing;
there are no prefilled reference answers. Repeat for the template, then continue to section 2.
You may also edit the CSV fields directly, but the training guard checks approval/provenance/word count.

The blind ID prevents the viewer from displaying labels, but the CSV `image_id` contains the category folder
for joining records. Avoid reading it while writing targets. Review status is a human declaration;
software cannot verify visual truth or certify the absence of all semantic label clues.

In [ ]:
REVIEWED_TARGET = ""  # Fill with your description of the image displayed above, 5-8 English words.
REVIEWER_ID = ""  # Your stable reviewer identifier, not a dataset label.
CAPTION_SOURCE = "human_corrected"  # human / human_corrected / teacher
TEACHER_MODEL = ""  # Required only for teacher proposals that you have personally reviewed.
if REVIEWED_TARGET.strip():
    save_caption_review(REVIEW_CSV, current_review_id, REVIEWED_TARGET, REVIEWER_ID, CAPTION_SOURCE, TEACHER_MODEL)
    print("Saved approved review:", current_review_id)
else:
    print("Nothing approved. Inspect an image and fill REVIEWED_TARGET and REVIEWER_ID before saving.")

## 2. Validate References Before Any Fine-Tuning

This is the intentional stopping point on the first run. Review all 188 default examples for balanced
coverage if feasible. The minimum of 50 is only a guard; fewer or narrow-quality references may hurt the model.
Both fit and caption holdout are subsets of **official train**, disjoint by image ID. Official val/test remain untouched.
Do not change reference captions after examining holdout results repeatedly without creating a new protocol.

In [ ]:
approved_pairs = approved_reference_pairs(REVIEW_CSV, caption_records, minimum=MIN_APPROVED_REFERENCES)
fit_pairs, holdout_pairs = split_reference_pairs(approved_pairs, REFERENCE_VAL_FRACTION, REVIEW_SEED)
print("Approved train-only references:", len(approved_pairs), "| fit:", len(fit_pairs), "| caption holdout:", len(holdout_pairs))
atomic_json(OUTPUT_DIR / "review" / "reference_split.json", {"fit": fit_pairs, "holdout": holdout_pairs})
lexical_overlap = sum(any(" " + normalized_caption(name) + " " in " " + normalized_caption(row["target_caption"]) + " "
                          for name in dtd_class_names) for row in approved_pairs)
print("References with direct DTD class-word overlap:", lexical_overlap,
      "(diagnostic; do not rewrite visually true references from class labels)")

## 3. Offline Training Runtime and Locked Protocol

Requires a GPU, the **legacy Microsoft Florence-2-base** folder already attached at
`/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi/Florence-2-base`, and compatible offline wheels.
The child process uses **Transformers 4.49.x + PEFT 0.14.0**; your kernel's Transformers 5.x is unchanged.
The old notebook input may not include PEFT: attach `peft-0.14.0-py3-none-any.whl` separately when needed.
Use compatible Python/platform wheels for Transformers, tokenizers, huggingface-hub and any missing dependency.
Wheel lookup is local only; this notebook never enables Internet or installs packages into the kernel environment.

Trainable parameters are strictly decoder attention `q_proj`/`v_proj` LoRA adapters, including decoder cross-attention.
References, image hashes, source code, model assets, settings and split are locked. Changed references/settings require
a **new output directory**. `last.pt` resumes the last completed epoch with optimizer/scheduler/scaler/RNG state;
an interrupted partial epoch is replayed. `best.pt` is selected only by caption holdout token NLL.
`best_adapter/` contains LoRA weights, not a standalone full Florence model.

In [ ]:
def florence_checkpoint(model_ref, roots):
    def checked(path):
        path = Path(path)
        try:
            config = json.loads((path / "config.json").read_text(encoding="utf-8"))
        except (OSError, ValueError):
            return None
        if config.get("model_type") != "florence2":
            return None
        indexes = list(path.glob("*.index.json"))
        weights = any((path / name).is_file() for name in ("model.safetensors", "pytorch_model.bin"))
        for index in indexes:
            try:
                shards = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())
                weights |= bool(shards) and all((path / shard).is_file() for shard in shards)
            except (OSError, ValueError):
                continue
        if not weights:
            return None
        backend = "legacy" if "AutoModelForCausalLM" in config.get("auto_map", {}) else "native"
        if backend == "legacy" and not all((path / name).is_file() for name in (
                "configuration_florence2.py", "modeling_florence2.py", "processing_florence2.py")):
            return None
        return path.resolve(), backend

    requested = Path(model_ref).expanduser()
    if requested.is_dir():
        result = checked(requested)
        if result:
            return result
        for config in sorted(requested.rglob("config.json")):
            result = checked(config.parent)
            if result:
                return result
        raise FileNotFoundError(f"Florence folder is incomplete: {requested}; need config, weights, processor/tokenizer and local model code.")
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git"})
            if len(Path(current).relative_to(root).parts) >= 8:
                dirs[:] = []
            if "config.json" in files:
                result = checked(current)
                if result:
                    return result
    raise FileNotFoundError(
        f"No complete local Florence-2 checkpoint found for {model_ref}. "
        "Attach notebook-mt-moi/Florence-2-base or set FLORENCE_MODEL_REF to the correct local folder. No download is attempted."
    )

def offline_caption_wheels(roots, destination):
    import zipfile
    destination = Path(destination)
    wheel_dirs = set()
    prefixes = ("peft-", "transformers-", "tokenizers-", "huggingface_hub-", "accelerate-", "einops-", "timm-")
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git", "Florence-2-base"})
            if len(Path(current).relative_to(root).parts) >= 7:
                dirs[:] = []
            for name in sorted(files):
                path = Path(current) / name
                if name.endswith(".whl"):
                    wheel_dirs.add(str(path.parent))
                elif name.lower() in {"offline_wheels.zip", "wheels.zip"}:
                    with zipfile.ZipFile(path) as archive:
                        for member in archive.infolist():
                            filename = Path(member.filename.replace("\\", "/")).name
                            if not filename.endswith(".whl") or not filename.startswith(prefixes):
                                continue
                            destination.mkdir(parents=True, exist_ok=True)
                            target = destination / filename
                            if not target.is_file() or target.stat().st_size != member.file_size:
                                with archive.open(member) as source, target.open("wb") as handle:
                                    shutil.copyfileobj(source, handle)
                            wheel_dirs.add(str(destination))
    return sorted(wheel_dirs)

def caption_runtime_environment(backend, asset_root, runtime_dir):
    import subprocess
    import sys
    runtime_dir = Path(runtime_dir)
    runtime_dir.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ)
    env.update(TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
               HF_MODULES_CACHE=str(runtime_dir / "hf_modules"), PYTHONUNBUFFERED="1")
    target = runtime_dir / "legacy_packages"
    probe = (
        "import transformers; "
        + ("assert transformers.__version__.startswith('4.49.'), transformers.__version__; " if backend == "legacy" else "")
        + "from transformers import AutoProcessor, AutoTokenizer, CLIPModel; "
        + ("from transformers import AutoModelForCausalLM; " if backend == "legacy" else
           "from transformers import Florence2ForConditionalGeneration; ")
        + "import einops, timm, accelerate; print(transformers.__version__)"
    )
    if target.is_dir():
        env["PYTHONPATH"] = str(target) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    result = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)
    if result.returncode == 0:
        print("Caption subprocess Transformers:", result.stdout.strip())
        return env
    if backend != "legacy":
        raise RuntimeError("The local native Florence checkpoint cannot load in this runtime. "
                           "Use a matching native Transformers installation or the attached legacy model + offline 4.49 wheels.\n"
                           + result.stderr[-3000:])
    wheel_dirs = offline_caption_wheels([Path(asset_root), Path("/kaggle/input")], runtime_dir / "wheels")
    if not wheel_dirs:
        raise FileNotFoundError(
            "The attached Microsoft Florence checkpoint needs Transformers 4.49.x. "
            f"No offline wheels found under {asset_root}. Attach offline_wheels.zip/wheels.zip or a wheel folder "
            "with transformers==4.49.0, tokenizers 0.21.x and huggingface-hub <1 compatible with this Python version. "
            "The notebook never enables Internet or modifies the kernel's Transformers."
        )
    command = [sys.executable, "-m", "pip", "install", "--no-index", "--only-binary=:all:",
               "--no-deps", "--upgrade", "--target", str(target), "transformers==4.49.0",
               "tokenizers>=0.21,<0.22", "huggingface-hub>=0.26,<1"]
    import importlib.util
    for module in ("einops", "timm", "accelerate"):
        if importlib.util.find_spec(module) is None:
            command.append(module)
    for directory in wheel_dirs:
        command.extend(["--find-links", directory])
    print("Preparing isolated Florence dependencies from attached offline wheels; the notebook kernel is unchanged.")
    installed = subprocess.run(command, env=env, capture_output=True, text=True)
    if installed.returncode:
        raise RuntimeError("Offline Florence dependency setup failed. Wheels must match Kaggle's Python/platform.\n"
                           + installed.stdout[-2000:] + installed.stderr[-3000:])
    base_pythonpath = os.environ.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(target) + (os.pathsep + base_pythonpath if base_pythonpath else "")
    checked = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)
    if checked.returncode:
        raise RuntimeError("The offline Florence environment still lacks a compatible dependency. "
                           "No kernel packages were changed.\n" + checked.stderr[-4000:])
    print("Isolated caption subprocess Transformers:", checked.stdout.strip())
    return env


def lora_runtime_environment(asset_root, runtime_dir):
    env = caption_runtime_environment("legacy", asset_root, runtime_dir)
    target = Path(runtime_dir) / "legacy_packages"
    probe = ("import transformers, peft; assert transformers.__version__.startswith('4.49.'); "
             "assert peft.__version__ == '0.14.0', peft.__version__; "
             "from peft import LoraConfig, PeftModel, get_peft_model; "
             "print('Transformers', transformers.__version__, '| PEFT', peft.__version__)")
    checked = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)
    if checked.returncode == 0:
        print(checked.stdout.strip())
        return env
    wheels = offline_caption_wheels([Path(asset_root), Path("/kaggle/input")], Path(runtime_dir) / "wheels")
    if not wheels:
        raise FileNotFoundError("Attach the offline peft==0.14.0 wheel, plus compatible Florence wheels. "
                                "Reviewing captions does not require PEFT. No Internet is enabled.")
    command = [sys.executable, "-m", "pip", "install", "--no-index", "--only-binary=:all:",
               "--no-deps", "--upgrade", "--target", str(target), "peft==0.14.0"]
    for folder in wheels:
        command.extend(["--find-links", folder])
    installed = subprocess.run(command, env=env, capture_output=True, text=True)
    if installed.returncode:
        raise RuntimeError("Offline PEFT setup failed: attach peft-0.14.0-py3-none-any.whl in a wheel folder "
                           "or offline_wheels.zip/wheels.zip. Kernel packages were not changed.\n"
                           + installed.stdout[-2000:] + installed.stderr[-3000:])
    target_prefix = str(target)
    if not env.get("PYTHONPATH", "").split(os.pathsep)[0] == target_prefix:
        env["PYTHONPATH"] = target_prefix + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    checked = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)
    if checked.returncode:
        raise RuntimeError("PEFT/Florence worker imports still fail; attach the missing compatible offline dependency.\n"
                           + checked.stderr[-4000:])
    print(checked.stdout.strip())
    return env


def run_caption_worker(action, payload_path, runtime_dir, env):
    log_path = Path(runtime_dir) / f"{action}.log"
    worker_path = Path(runtime_dir) / "finetune_florence_dtd.py"
    worker_path.write_text(CAPTION_FINETUNE_WORKER_SOURCE, encoding="utf-8")
    with log_path.open("a", encoding="utf-8") as log:
        process = subprocess.Popen([sys.executable, str(worker_path), str(payload_path), action], env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
        finally:
            process.stdout.close()
    if code:
        raise RuntimeError(f"Caption {action} failed. Inspect {log_path}; completed checkpoints/CSV batches are preserved.")


CAPTION_FINETUNE_WORKER_SOURCE = '"""Review-first supervised Florence caption adaptation; embedded in a Kaggle notebook."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport re\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path, PurePosixPath\n\ndef clean_dtd_caption(text):\n    import re\n    text = str(text).strip()\n    for _ in range(3):\n        before = text\n        text = re.sub(r"^(?:(?:the|this)\\s+)?(?:image|picture|photo(?:graph)?)\\s+"\n                      r"(?:is|shows|depicts|features|contains)\\s+", "", text, flags=re.I)\n        text = re.sub(r"^(?:(?:a|an|the)\\s+)?close[ -]up(?:\\s+view)?\\s+(?:of|showing)\\s+",\n                      "", text, flags=re.I)\n        text = re.sub(r"^(?:a|an|the)(?:\\s+|$)", "", text, flags=re.I)\n        if text == before:\n            break\n    words = re.findall(r"[A-Za-z0-9]+(?:[\'\\u2019][A-Za-z0-9]+)?", text)\n    selected = []\n    normalized_count = 0\n    for word in words:\n        count = len(re.findall(r"[A-Za-z0-9]+", word))\n        if normalized_count + count > 8:\n            break\n        selected.append(word)\n        normalized_count += count\n    return " ".join(selected)\n\n\nREFERENCE_COLUMNS = ["review_id", "image_id", "draft_caption_unverified", "target_caption",\n                     "caption_source", "review_status", "reviewer_id", "teacher_model", "notes"]\nEXPORT_COLUMNS = ["image_id", "split", "raw_caption", "final_caption", "caption_valid",\n    "leakage_detected", "lexical_leakage_detected", "semantic_leakage_detected",\n    "semantic_leakage_score", "semantic_leakage_class", "retry_count", "vlm_model", "prompt_version",\n    "caption_input_policy", "semantic_audit_status", "captioner_fingerprint"]\nTASK = "<MORE_DETAILED_CAPTION>"\nPROMPT_VERSION = "florence2_dtd_decoder_lora_short_v1"\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef object_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,\n                                    allow_nan=False).encode()).hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef read_csv_rows(path, required=()):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):\n            raise ValueError(f"Missing/duplicate CSV header: {path}")\n        if not set(required).issubset(reader.fieldnames):\n            raise ValueError(f"Missing required CSV columns: {sorted(set(required) - set(reader.fieldnames))}")\n        rows = list(reader)\n    if any(None in row or any(value is None for value in row.values()) for row in rows):\n        raise ValueError(f"Malformed/incomplete CSV row: {path}; original file was not modified")\n    return rows\n\n\ndef atomic_csv(path, rows, columns):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n        handle.flush()\n        os.fsync(handle.fileno())\n    temporary.replace(path)\n\n\ndef normalized_caption(text):\n    return re.sub(r"[^a-z0-9]+", " ", str(text).lower().replace("_", " ")).strip()\n\n\ndef load_caption_manifest(root, split, expected_classes=47, per_class=40):\n    root = Path(root).expanduser().resolve()\n    if not 1 <= int(split) <= 10:\n        raise ValueError("DTD_SPLIT must be in 1..10")\n    images = (root / "images").resolve()\n    names = sorted(path.name for path in images.iterdir() if path.is_dir())\n    if len(names) != expected_classes:\n        raise ValueError(f"Expected {expected_classes} DTD image folders; found {len(names)} at {images}")\n    records, seen = [], set()\n    for role in ("train", "val", "test"):\n        counts = defaultdict(int)\n        for raw in (root / "labels" / f"{role}{split}.txt").read_text(encoding="utf-8-sig").splitlines():\n            text = raw.strip().replace("\\\\", "/")\n            if not text:\n                continue\n            parts = PurePosixPath(text).parts\n            if len(parts) != 2 or parts[0] not in names or any(part in {"..", "."} for part in parts):\n                raise ValueError(f"Unexpected official DTD path: {text!r}")\n            image_id = PurePosixPath(*parts).as_posix()\n            path = images.joinpath(*parts).resolve()\n            if not path.is_relative_to(images) or not path.is_file():\n                raise FileNotFoundError(f"DTD image missing/outside images root: {path}")\n            if image_id in seen:\n                raise ValueError(f"Duplicate image or overlapping official splits: {image_id}")\n            seen.add(image_id)\n            counts[parts[0]] += 1\n            records.append({"image_id": image_id, "image_path": str(path), "split": role})\n        if set(counts) != set(names) or any(count != per_class for count in counts.values()):\n            raise ValueError(f"Official {role}{split}.txt must contain {per_class} images/class")\n    return records, names\n\n\ndef create_reference_template(records, output, per_class=4, seed=42, draft_csv=""):\n    output = Path(output)\n    if per_class < 1:\n        raise ValueError("Review sample size per class must be positive")\n    train = {row["image_id"]: row for row in records if row["split"] == "train"}\n    if output.is_file():\n        existing = read_csv_rows(output, REFERENCE_COLUMNS)\n        if any(row.get("image_id") not in train for row in existing):\n            raise ValueError("Existing review template has non-training images; use a separate output directory")\n        return output  # Never overwrite manual review work on rerun.\n    drafts = {}\n    if draft_csv:\n        for row in read_csv_rows(draft_csv):\n            if row.get("image_id") in train:\n                drafts[row["image_id"]] = row.get("raw_caption", "").strip() or row.get("final_caption", "")\n    by_class = defaultdict(list)\n    for image_id in train:\n        by_class[image_id.split("/")[0]].append(image_id)\n    rows = []\n    for category in sorted(by_class):\n        ordered = sorted(by_class[category], key=lambda key: object_hash([seed, key]))\n        for image_id in ordered[:per_class]:\n            rows.append({"review_id": object_hash(["blind-review", image_id])[:12], "image_id": image_id,\n                "draft_caption_unverified": drafts.get(image_id, ""), "target_caption": "",\n                "caption_source": "", "review_status": "pending", "reviewer_id": "",\n                "teacher_model": "", "notes": ""})\n    rows.sort(key=lambda row: object_hash(["review-order", seed, row["review_id"]]))\n    atomic_csv(output, rows, REFERENCE_COLUMNS)\n    return output\n\n\ndef validate_reference_text(text):\n    text = " ".join(str(text).split())\n    count = len(normalized_caption(text).split())\n    if not 5 <= count <= 8:\n        raise ValueError("Approved target_caption must contain 5-8 normalized English words; do not truncate references silently")\n    if normalized_caption(clean_dtd_caption(text)) != normalized_caption(text):\n        raise ValueError("Remove generic image/close-up openings from the reviewed target caption")\n    return text\n\n\ndef save_caption_review(template, review_id, caption, reviewer_id, source="human_corrected", teacher_model=""):\n    if not str(reviewer_id).strip():\n        raise ValueError("Supply the human reviewer_id; approval is a provenance declaration, not an automated correctness check")\n    if source not in {"human", "human_corrected", "teacher"}:\n        raise ValueError("caption_source must be human, human_corrected or teacher")\n    if source == "teacher" and not teacher_model.strip():\n        raise ValueError("Record teacher_model for teacher captions, even after human review")\n    target = validate_reference_text(caption)\n    rows = read_csv_rows(template)\n    matches = [row for row in rows if row.get("review_id") == review_id]\n    if len(matches) != 1:\n        raise ValueError("review_id must identify exactly one template row")\n    matches[0].update(target_caption=target, reviewer_id=reviewer_id.strip(), caption_source=source,\n                      review_status="approved", teacher_model=teacher_model.strip())\n    atomic_csv(template, rows, REFERENCE_COLUMNS)\n\n\ndef approved_reference_pairs(path, records, minimum=50):\n    train = {row["image_id"]: row for row in records if row["split"] == "train"}\n    rows = read_csv_rows(path)\n    required = {"image_id", "target_caption", "caption_source", "review_status", "reviewer_id"}\n    if not rows or not required.issubset(rows[0]):\n        raise ValueError("Reference CSV must contain image_id,target_caption,caption_source,review_status,reviewer_id")\n    seen, approved = set(), []\n    for row in rows:\n        image_id = row["image_id"].strip()\n        if image_id not in train:\n            raise ValueError(f"Only official TRAIN images may be reference examples: {image_id}")\n        if image_id in seen:\n            raise ValueError(f"Duplicate reference image_id: {image_id}")\n        seen.add(image_id)\n        if row["review_status"].strip().lower() != "approved":\n            continue\n        source = row["caption_source"].strip().lower()\n        if source not in {"human", "human_corrected", "teacher"} or not row["reviewer_id"].strip():\n            raise ValueError(f"Approved reference lacks human review provenance: {image_id}")\n        if source == "teacher" and not row.get("teacher_model", "").strip():\n            raise ValueError(f"Teacher caption requires teacher_model and human approval: {image_id}")\n        approved.append({**train[image_id], "target_caption": validate_reference_text(row["target_caption"]),\n            "caption_source": source, "reviewer_id": row["reviewer_id"].strip(),\n            "teacher_model": row.get("teacher_model", ""), "image_sha256": file_sha256(train[image_id]["image_path"])})\n    if len(approved) < minimum:\n        raise ValueError(f"Only {len(approved)} approved caption references; need at least {minimum}. "\n                         "Review the template and save captions first. Florence drafts are NOT ground truth; "\n                         "no model was trained and no reference answers were invented.")\n    return sorted(approved, key=lambda row: row["image_id"])\n\n\ndef split_reference_pairs(pairs, fraction=.20, seed=42):\n    if not 0 < fraction < .5:\n        raise ValueError("Reference validation fraction must be in (0, 0.5)")\n    groups = defaultdict(list)\n    for row in pairs:\n        if row["split"] != "train":\n            raise ValueError("Captioner reference train/holdout must both come from official DTD train")\n        groups[row["image_id"].split("/")[0]].append(row)\n    fit, holdout = [], []\n    for group in groups.values():\n        group = sorted(group, key=lambda row: object_hash([seed, row["image_id"]]))\n        count = min(max(1, round(len(group) * fraction)), len(group) - 1) if len(group) > 1 else 0\n        holdout.extend(group[:count])\n        fit.extend(group[count:])\n    if len(holdout) < 2 or not fit:\n        raise ValueError("Need at least two held-out reference images; approve more training images per class")\n    return sorted(fit, key=lambda row: row["image_id"]), sorted(holdout, key=lambda row: row["image_id"])\n\n\ndef checkpoint_asset_identity(model_dir):\n    model_dir = Path(model_dir)\n    files = sorted(path for path in model_dir.iterdir() if path.is_file() and\n                   (path.suffix in {".py", ".json", ".safetensors", ".bin", ".txt"}))\n    if not any(path.suffix in {".safetensors", ".bin"} for path in files):\n        raise ValueError("Checkpoint has no weight files")\n    return {path.name: file_sha256(path) for path in files}\n\n\ndef lock_caption_protocol(output, protocol):\n    output = Path(output)\n    path = output / "caption_training_protocol.json"\n    if path.is_file() and json.loads(path.read_text(encoding="utf-8")) != protocol:\n        raise ValueError("Changed caption references, assets, code or hyperparameters: choose a new output directory")\n    if not path.is_file() and any((output / name).exists() for name in ("last.pt", "best.pt", "best_adapter")):\n        raise ValueError("Existing checkpoint lacks this run\'s protocol; do not adopt another run\'s weights")\n    atomic_json(path, protocol)\n    return object_hash(protocol)\n\n\ndef seed_caption_run(seed):\n    import numpy as np\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef validate_caption_payload(payload):\n    settings = payload["settings"]\n    for name in ("epochs", "batch_size", "accumulation", "lora_rank", "lora_alpha",\n                 "max_target_tokens", "max_new_tokens", "export_batch_size"):\n        if type(settings[name]) is not int or settings[name] < 1:\n            raise ValueError(f"{name} must be a positive integer")\n    if not math.isfinite(settings["learning_rate"]) or settings["learning_rate"] <= 0:\n        raise ValueError("learning_rate must be finite and positive")\n    if not math.isfinite(settings["weight_decay"]) or settings["weight_decay"] < 0:\n        raise ValueError("weight_decay must be finite and nonnegative")\n    if not 0 <= settings["lora_dropout"] < 1:\n        raise ValueError("lora_dropout must be in [0, 1)")\n    for key in ("settings", "fit", "holdout"):\n        if payload[key] != payload["protocol"][key]:\n            raise ValueError(f"Worker {key} does not match the locked training protocol")\n    seen = set()\n    for row in payload["fit"] + payload["holdout"]:\n        if row["split"] != "train" or row["image_id"] in seen:\n            raise ValueError("Reference sets must be disjoint and contain official training images only")\n        seen.add(row["image_id"])\n        validate_reference_text(row["target_caption"])\n        if file_sha256(row["image_path"]) != row["image_sha256"]:\n            raise ValueError("Reference image changed since review/protocol preparation")\n    if not payload["fit"] or len(payload["holdout"]) < 2:\n        raise ValueError("Prepare approved reference fit and holdout sets first")\n\n\nclass CaptionPairDataset:\n    def __init__(self, pairs):\n        self.pairs = pairs\n\n    def __len__(self):\n        return len(self.pairs)\n\n    def __getitem__(self, index):\n        from PIL import Image, ImageOps\n        row = self.pairs[index]\n        with Image.open(row["image_path"]) as source:\n            image = ImageOps.exif_transpose(source).convert("RGB")\n        return image, row["target_caption"]\n\n\ndef caption_collate(items, processor, max_target_tokens=48):\n    import torch\n    images, targets = zip(*items)\n    inputs = processor(text=[TASK] * len(images), images=list(images), padding=True, return_tensors="pt")\n    tokens = processor.tokenizer(list(targets), padding=True, truncation=False, return_tensors="pt")\n    if tokens["input_ids"].shape[1] > max_target_tokens:\n        raise ValueError("Reference token sequence exceeds max_target_tokens; inspect targets rather than truncating them")\n    labels = tokens["input_ids"].clone()\n    labels[tokens["attention_mask"] == 0] = -100\n    if not torch.any(labels != -100):\n        raise ValueError("Empty reference-token batch")\n    return {key: value for key, value in inputs.items() if torch.is_tensor(value)}, labels\n\n\ndef decoder_lora_targets(model):\n    import torch\n    targets = [name for name, module in model.named_modules() if name.startswith("language_model.")\n               and ".decoder." in name and name.rsplit(".", 1)[-1] in {"q_proj", "v_proj"}\n               and isinstance(module, torch.nn.Linear)]\n    if not targets:\n        raise ValueError("No Florence decoder q_proj/v_proj modules found; do not silently adapt the vision encoder")\n    return sorted(targets)\n\n\ndef load_caption_model(payload, device, trainable=False):\n    import torch\n    from transformers import AutoModelForCausalLM, AutoProcessor\n    from peft import LoraConfig, PeftModel, get_peft_model\n    processor = AutoProcessor.from_pretrained(payload["model"], trust_remote_code=True, local_files_only=True)\n    model = AutoModelForCausalLM.from_pretrained(payload["model"], trust_remote_code=True,\n        local_files_only=True, torch_dtype=torch.float32, low_cpu_mem_usage=True, attn_implementation="eager")\n    if trainable:\n        for parameter in model.parameters():\n            parameter.requires_grad_(False)\n        targets = decoder_lora_targets(model)\n        settings = payload["settings"]\n        # Generic PEFT preserves Florence\'s image-aware forward/generate signature.\n        model = get_peft_model(model, LoraConfig(r=settings["lora_rank"], lora_alpha=settings["lora_alpha"],\n            lora_dropout=settings["lora_dropout"], target_modules=targets, bias="none"))\n        names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]\n        if not names or any("lora_" not in name or ".decoder." not in name for name in names):\n            raise ValueError("Unexpected trainable parameters; vision encoder/base model must remain frozen")\n        atomic_json(Path(payload["output"]) / "trainable_parameters.json", {\n            "targets": targets, "parameters": names,\n            "trainable_count": sum(p.numel() for p in model.parameters() if p.requires_grad),\n            "total_count": sum(p.numel() for p in model.parameters())})\n    else:\n        model = PeftModel.from_pretrained(model, str(Path(payload["output"]) / "best_adapter"),\n                                         is_trainable=False, local_files_only=True)\n    return model.to(device), processor\n\n\ndef freeze_caption_vision(model):\n    base = model.get_base_model()\n    if hasattr(base, "vision_tower"):\n        base.vision_tower.eval()\n\n\ndef adapter_state(model):\n    from peft import get_peft_model_state_dict\n    return {name: tensor.detach().cpu().clone() for name, tensor in get_peft_model_state_dict(model).items()}\n\n\ndef restore_adapter(model, state):\n    from peft import set_peft_model_state_dict\n    set_peft_model_state_dict(model, state)\n\n\ndef atomic_torch_file(path, state):\n    import torch\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    torch.save(state, temporary)\n    temporary.replace(path)\n\n\ndef capture_caption_rng(generator):\n    import numpy as np\n    import torch\n    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),\n            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n            "loader": generator.get_state()}\n\n\ndef restore_caption_rng(state, generator):\n    import numpy as np\n    import torch\n    random.setstate(state["python"])\n    np.random.set_state(state["numpy"])\n    torch.set_rng_state(state["torch"])\n    if state["cuda"]:\n        torch.cuda.set_rng_state_all(state["cuda"])\n    generator.set_state(state["loader"])\n\n\ndef move_caption_batch(batch, device):\n    inputs, labels = batch\n    return {key: value.to(device) for key, value in inputs.items()}, labels.to(device)\n\n\ndef caption_autocast(device):\n    import torch\n    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16\n    return torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda")\n\n\ndef evaluate_caption_nll(model, loader, device):\n    import torch\n    model.eval()\n    total, tokens = 0., 0\n    with torch.inference_mode():\n        for batch in loader:\n            inputs, labels = move_caption_batch(batch, device)\n            count = int((labels != -100).sum())\n            with caption_autocast(device):\n                loss = model(**inputs, labels=labels, use_cache=False).loss\n            if not torch.isfinite(loss):\n                raise ValueError("Non-finite caption validation NLL")\n            total += float(loss) * count\n            tokens += count\n    return total / max(tokens, 1)\n\n\ndef train_caption_epoch(model, loader, optimizer, scheduler, scaler, device, accumulation):\n    import torch\n    model.train()\n    freeze_caption_vision(model)\n    optimizer.zero_grad(set_to_none=True)\n    total_loss = total_tokens = group_tokens = 0\n    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]\n    for index, batch in enumerate(loader):\n        inputs, labels = move_caption_batch(batch, device)\n        count = int((labels != -100).sum())\n        with caption_autocast(device):\n            loss = model(**inputs, labels=labels, use_cache=False).loss\n        if not torch.isfinite(loss):\n            raise ValueError("Non-finite caption training loss; last completed epoch is preserved")\n        scaler.scale(loss * count).backward()\n        total_loss += float(loss.detach()) * count\n        total_tokens += count\n        group_tokens += count\n        if (index + 1) % accumulation == 0 or index + 1 == len(loader):\n            scaler.unscale_(optimizer)\n            for parameter in parameters:\n                if parameter.grad is not None:\n                    parameter.grad.div_(group_tokens)\n            torch.nn.utils.clip_grad_norm_(parameters, 1.)\n            previous_scale = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            if scaler.get_scale() >= previous_scale:\n                scheduler.step()\n            optimizer.zero_grad(set_to_none=True)\n            group_tokens = 0\n    return total_loss / max(total_tokens, 1)\n\n\ndef train_captioner(payload, device):\n    import torch\n    from functools import partial\n    from torch.utils.data import DataLoader\n    validate_caption_payload(payload)\n    settings = payload["settings"]\n    output = Path(payload["output"])\n    seed_caption_run(settings["seed"])\n    fingerprint = lock_caption_protocol(output, payload["protocol"])\n    model, processor = load_caption_model(payload, device, trainable=True)\n    generator = torch.Generator().manual_seed(settings["seed"])\n    collate = partial(caption_collate, processor=processor, max_target_tokens=settings["max_target_tokens"])\n    train_loader = DataLoader(CaptionPairDataset(payload["fit"]), batch_size=settings["batch_size"],\n        shuffle=True, generator=generator, num_workers=0, collate_fn=collate)\n    val_loader = DataLoader(CaptionPairDataset(payload["holdout"]), batch_size=settings["batch_size"],\n        shuffle=False, num_workers=0, collate_fn=collate)\n    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],\n                                  lr=settings["learning_rate"], weight_decay=settings["weight_decay"])\n    updates_per_epoch = math.ceil(len(train_loader) / settings["accumulation"])\n    total_updates = updates_per_epoch * settings["epochs"]\n    warmup = max(1, int(total_updates * .1))\n    def factor(step):\n        if step < warmup:\n            return (step + 1) / warmup\n        return max(.01, .5 * (1 + math.cos(math.pi * (step - warmup) / max(total_updates - warmup, 1))))\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, factor)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())\n    history, start, best_nll, best_epoch = [], 1, float("inf"), 0\n    resume_path, best_path = output / "last.pt", output / "best.pt"\n    if resume_path.is_file():\n        state = torch.load(resume_path, map_location="cpu", weights_only=False)\n        if state["identity"] != fingerprint:\n            raise ValueError("Resume checkpoint belongs to another caption experiment")\n        restore_adapter(model, state["adapter"])\n        optimizer.load_state_dict(state["optimizer"])\n        scheduler.load_state_dict(state["scheduler"])\n        scaler.load_state_dict(state["scaler"])\n        history, best_nll, best_epoch = state["history"], state["best_nll"], state["best_epoch"]\n        start = state["epoch"] + 1\n        restore_caption_rng(state["rng"], generator)\n        print("Resuming captioner at epoch", start, flush=True)\n    # DataLoader iteration consumes RNG even without shuffling; preserve exact epoch-boundary resume.\n    baseline_rng = capture_caption_rng(generator)\n    with model.disable_adapter():\n        baseline_nll = evaluate_caption_nll(model, val_loader, device)\n    restore_caption_rng(baseline_rng, generator)\n    for epoch in range(start, settings["epochs"] + 1):\n        training = train_caption_epoch(model, train_loader, optimizer, scheduler, scaler, device, settings["accumulation"])\n        validation = evaluate_caption_nll(model, val_loader, device)\n        history.append({"epoch": epoch, "train_token_nll": training, "holdout_token_nll": validation})\n        if validation < best_nll:\n            best_nll, best_epoch = validation, epoch\n            atomic_torch_file(best_path, {"identity": fingerprint, "epoch": epoch, "holdout_nll": validation,\n                                         "adapter": adapter_state(model)})\n        atomic_torch_file(resume_path, {"identity": fingerprint, "epoch": epoch, "adapter": adapter_state(model),\n            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),\n            "history": history, "best_nll": best_nll, "best_epoch": best_epoch, "rng": capture_caption_rng(generator)})\n        atomic_csv(output / "history.csv", history, ["epoch", "train_token_nll", "holdout_token_nll"])\n        print(f"Caption epoch {epoch}/{settings[\'epochs\']}: train_nll={training:.4f}; holdout_nll={validation:.4f}", flush=True)\n    best = torch.load(best_path, map_location="cpu", weights_only=False)\n    if best["identity"] != fingerprint:\n        raise ValueError("Best adapter has the wrong identity")\n    restore_adapter(model, best["adapter"])\n    adapter_dir = output / "best_adapter"\n    model.save_pretrained(adapter_dir, safe_serialization=True, save_embedding_layers=False)\n    processor.save_pretrained(output / "processor")\n    comparisons = []\n    for row in payload["holdout"]:\n        with model.disable_adapter():\n            before = generate_caption_texts(model, processor, [row], device, settings["max_new_tokens"])[0]\n        after = generate_caption_texts(model, processor, [row], device, settings["max_new_tokens"])[0]\n        comparisons.append({"image_id": row["image_id"], "reference": row["target_caption"],\n            "base_raw_caption": before, "adapted_raw_caption": after,\n            "base_short_caption": clean_dtd_caption(before), "adapted_short_caption": clean_dtd_caption(after),\n            "human_preference": "", "review_notes": ""})\n    comparison_path = output / "holdout_before_after.csv"\n    if comparison_path.is_file():\n        previous = {row["image_id"]: row for row in read_csv_rows(comparison_path)}\n        for row in comparisons:\n            old = previous.get(row["image_id"], {})\n            if any(old.get(key) != value for key, value in row.items()\n                   if key not in {"human_preference", "review_notes"}):\n                raise ValueError("Existing holdout comparison differs; preserve manual reviews and use a new output directory")\n        # Keep the original file, including human preferences, notes and any extra review columns.\n    else:\n        atomic_csv(comparison_path, comparisons, list(comparisons[0]))\n    summary = {"identity": fingerprint, "epochs_completed": len(history), "best_epoch": best_epoch,\n        "baseline_holdout_token_nll": baseline_nll, "best_holdout_token_nll": best_nll,\n        "fit_images": len(payload["fit"]), "holdout_images": len(payload["holdout"]),\n        "reference_origin": "human_reviewed_train_only", "downstream_f1_improvement_verified": False}\n    atomic_json(output / "training_complete.json", summary)\n    print(json.dumps(summary, indent=2), flush=True)\n    return summary\n\n\ndef generate_caption_texts(model, processor, rows, device, max_new_tokens):\n    import torch\n    from PIL import Image, ImageOps\n    images = []\n    for row in rows:\n        with Image.open(row["image_path"]) as source:\n            images.append(ImageOps.exif_transpose(source).convert("RGB"))\n    inputs = processor(text=[TASK] * len(images), images=images, padding=True, return_tensors="pt")\n    moved = {key: value.to(device) for key, value in inputs.items() if torch.is_tensor(value)}\n    model.eval()\n    with torch.inference_mode(), caption_autocast(device):\n        generated = model.generate(**moved, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False, use_cache=True)\n    decoded = processor.batch_decode(generated, skip_special_tokens=False)\n    if len(decoded) != len(rows):\n        raise ValueError("Florence returned the wrong number of captions")\n    results = []\n    for text, image in zip(decoded, images):\n        parsed = processor.post_process_generation(text, task=TASK, image_size=(image.width, image.height))\n        results.append(str(parsed.get(TASK, text) if isinstance(parsed, dict) else text).strip())\n    return results\n\n\nclass CaptionSemanticAuditor:\n    def __init__(self, path, names, device):\n        import torch\n        from transformers import CLIPTextModelWithProjection, CLIPTokenizer\n        if not Path(path).is_dir():\n            raise FileNotFoundError("Attach local CLIP ViT-B/32 for diagnostic audit, or explicitly disable run_clip_audit")\n        self.tokenizer = CLIPTokenizer.from_pretrained(path, local_files_only=True)\n        self.model = CLIPTextModelWithProjection.from_pretrained(path, local_files_only=True).to(device).eval()\n        self.names, self.device = names, device\n        prompts = [template.format(name=name) for name in names\n                   for template in ("a photo of {name}", "a scene showing {name}")]\n        vectors = self.encode(prompts).reshape(len(names), 2, -1).mean(1)\n        self.vectors = torch.nn.functional.normalize(vectors, dim=1)\n\n    def encode(self, texts):\n        import torch\n        vectors = []\n        with torch.inference_mode():\n            for start in range(0, len(texts), 128):\n                tokens = self.tokenizer(texts[start:start + 128], return_tensors="pt", padding=True,\n                    truncation=True, max_length=self.model.config.max_position_embeddings).to(self.device)\n                vectors.append(torch.nn.functional.normalize(self.model(**tokens).text_embeds.float(), dim=1))\n        return torch.cat(vectors)\n\n    def scores(self, texts):\n        scores, indices = (self.encode(texts) @ self.vectors.T).max(1)\n        return [(float(score), self.names[int(index)]) for score, index in zip(scores.cpu(), indices.cpu())]\n\n\ndef make_export_row(record, raw, identity, names, semantic=None):\n    final = clean_dtd_caption(raw)\n    normalized = " " + normalized_caption(final) + " "\n    lexical = any(" " + normalized_caption(name) + " " in normalized for name in names)\n    valid = 5 <= len(normalized.split()) <= 8 and not lexical\n    score, matched = semantic if semantic is not None else ("", "")\n    return {"image_id": record["image_id"], "split": record["split"], "raw_caption": raw,\n        "final_caption": final, "caption_valid": valid, "leakage_detected": lexical,\n        "lexical_leakage_detected": lexical, "semantic_leakage_detected": bool(semantic is not None and score >= .30),\n        "semantic_leakage_score": score, "semantic_leakage_class": matched, "retry_count": 0,\n        "vlm_model": "microsoft/Florence-2-base+dtd-decoder-lora:" + identity[:16],\n        "prompt_version": PROMPT_VERSION, "caption_input_policy": "dtd_lexical_v2",\n        "semantic_audit_status": "diagnostic_only" if semantic is not None else "not_run",\n        "captioner_fingerprint": identity}\n\n\ndef export_caption_csv(payload, device):\n    import torch\n    output = Path(payload["output"])\n    summary = json.loads((output / "training_complete.json").read_text(encoding="utf-8"))\n    training_id = object_hash(payload["protocol"])\n    if summary["identity"] != training_id or summary["epochs_completed"] != payload["settings"]["epochs"]:\n        raise ValueError("Finish this caption training protocol before exporting all DTD captions")\n    records = payload["export_records"]\n    if len({row["image_id"] for row in records}) != len(records):\n        raise ValueError("Duplicate export image IDs")\n    print("Hashing export images to verify dataset content across resumed sessions...", flush=True)\n    inventory = [{"image_id": row["image_id"], "split": row["split"],\n                  "sha256": file_sha256(row["image_path"])} for row in records]\n    manifest = {"training": training_id, "best_adapter": checkpoint_asset_identity(output / "best_adapter"),\n        "images": inventory, "prompt": TASK, "version": PROMPT_VERSION, "policy": "dtd_lexical_v2",\n        "max_new_tokens": payload["settings"]["max_new_tokens"], "sampling": "greedy",\n        "audit_class_names": list(payload["class_names"]),\n        "clip": checkpoint_asset_identity(payload["clip"]) if payload["run_clip_audit"] else None,\n        "clip_threshold": .30, "semantic_hard_gate": False}\n    identity = object_hash(manifest)\n    folder, csv_path = output / "exports", output / "exports" / "captions.csv"\n    manifest_path = folder / "caption_generation_manifest.json"\n    if manifest_path.is_file() and json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:\n        raise ValueError("Changed caption generation assets/options; use a new export directory, never mix CSV rows")\n    if csv_path.exists() and not manifest_path.exists():\n        raise ValueError("Existing captions.csv has no matching generation manifest; original file was not modified")\n    atomic_json(manifest_path, manifest)\n    existing = read_csv_rows(csv_path, EXPORT_COLUMNS) if csv_path.is_file() and csv_path.stat().st_size else []\n    expected = {row["image_id"]: row["split"] for row in records}\n    seen = set()\n    for row in existing:\n        if not set(EXPORT_COLUMNS).issubset(row) or row["image_id"] in seen or row["image_id"] not in expected:\n            raise ValueError("Invalid schema, duplicate or unexpected image in export CSV")\n        if row["captioner_fingerprint"] != identity or row["split"] != expected[row["image_id"]]:\n            raise ValueError("Export CSV belongs to another captioner/split")\n        seen.add(row["image_id"])\n    todo = [row for row in records if row["image_id"] not in seen]\n    print(f"Improved captions: cached={len(seen)}, remaining={len(todo)}, output={csv_path}", flush=True)\n    if todo:\n        model, processor = load_caption_model(payload, device, trainable=False)\n        auditor = CaptionSemanticAuditor(payload["clip"], payload["class_names"], device) if payload["run_clip_audit"] else None\n        size, offset = payload["settings"]["export_batch_size"], 0\n        while offset < len(todo):\n            group = todo[offset:offset + size]\n            try:\n                raw = generate_caption_texts(model, processor, group, device, payload["settings"]["max_new_tokens"])\n                scores = auditor.scores([clean_dtd_caption(text) for text in raw]) if auditor else [None] * len(group)\n            except torch.cuda.OutOfMemoryError:\n                if size == 1:\n                    raise\n                size = max(1, size // 2)\n                torch.cuda.empty_cache()\n                print("Export batch reduced to", size, flush=True)\n                continue\n            if len(raw) != len(group) or len(scores) != len(group):\n                raise ValueError("Caption/audit batch count mismatch; no rows from this batch were appended")\n            rows = [make_export_row(row, text, identity, payload["class_names"], score)\n                    for row, text, score in zip(group, raw, scores)]\n            header = not csv_path.exists() or csv_path.stat().st_size == 0\n            with csv_path.open("a", encoding="utf-8", newline="") as handle:\n                writer = csv.DictWriter(handle, fieldnames=EXPORT_COLUMNS)\n                if header:\n                    writer.writeheader()\n                writer.writerows(rows)\n                handle.flush()\n                os.fsync(handle.fileno())\n            offset += len(group)\n            print(f"Exported {len(seen) + offset}/{len(records)}", flush=True)\n    final = read_csv_rows(csv_path)\n    if {row["image_id"] for row in final} != set(expected):\n        raise ValueError("Export is incomplete; rerun export to resume missing images")\n    atomic_json(folder / "export_summary.json", {"rows": len(final), "csv_sha256": file_sha256(csv_path),\n        "captioner_fingerprint": identity, "policy": "dtd_lexical_v2",\n        "valid_captions": sum(row["caption_valid"] == "True" for row in final),\n        "human_ground_truth": False, "downstream_improvement_verified": False})\n    return csv_path\n\n\ndef caption_worker_main(payload_path, action):\n    import torch\n    import transformers\n    import peft\n    if not transformers.__version__.startswith("4.49.") or peft.__version__ != "0.14.0":\n        raise RuntimeError("This legacy Florence training notebook uses Transformers 4.49.x and PEFT 0.14.0 in its isolated worker")\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Kaggle GPU for real Florence training/export; CPU mocks are only for tests")\n    payload = json.loads(Path(payload_path).read_text(encoding="utf-8"))\n    if checkpoint_asset_identity(payload["model"]) != payload["protocol"]["base_assets"]:\n        raise ValueError("Base Florence assets changed after protocol preparation; do not reuse this run")\n    device = torch.device("cuda")\n    if action == "train":\n        train_captioner(payload, device)\n    elif action == "export":\n        export_caption_csv(payload, device)\n    else:\n        raise ValueError("Worker action must be train or export")\n\n\nif __name__ == "__main__":\n    caption_worker_main(sys.argv[1], sys.argv[2])\n'

In [ ]:
# The reference guard is repeated so this cell cannot bypass manual review after a kernel restart.
approved_pairs = approved_reference_pairs(REVIEW_CSV, caption_records, minimum=MIN_APPROVED_REFERENCES)
fit_pairs, holdout_pairs = split_reference_pairs(approved_pairs, REFERENCE_VAL_FRACTION, REVIEW_SEED)
model_dir, backend = florence_checkpoint(FLORENCE_MODEL_REF, [Path(OFFLINE_ASSETS_ROOT), Path("/kaggle/input")])
if backend != "legacy":
    raise ValueError("This fine-tuning notebook expects the attached legacy Microsoft Florence checkpoint, not native community weights")
print("Hashing the base checkpoint and reviewed images to lock the experiment...")
protocol = {
    "schema": "dtd_caption_review_lora_v1", "dtd_split": DTD_SPLIT, "seed_reference_split": REVIEW_SEED,
    "reference_val_fraction": REFERENCE_VAL_FRACTION,
    "settings": CAPTION_SETTINGS, "fit": fit_pairs, "holdout": holdout_pairs,
    "base_assets": checkpoint_asset_identity(model_dir), "prompt": TASK,
    "source_sha256": object_hash(CAPTION_FINETUNE_WORKER_SOURCE),
    "reference_policy": "human_reviewed_train_only_5_to_8_words", "export_policy": "dtd_lexical_v2",
}
protocol_id = lock_caption_protocol(OUTPUT_DIR, protocol)
payload = {"model": str(model_dir), "output": str(OUTPUT_DIR.resolve()), "protocol": protocol,
    "settings": CAPTION_SETTINGS, "fit": fit_pairs, "holdout": holdout_pairs,
    "export_records": caption_records, "class_names": dtd_class_names,
    "clip": CLIP_MODEL_LOCAL_PATH, "run_clip_audit": RUN_CLIP_AUDIT}
validate_caption_payload(payload)
RUNTIME_DIR = OUTPUT_DIR / "caption_runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
PAYLOAD_PATH = RUNTIME_DIR / "request.json"
atomic_json(PAYLOAD_PATH, payload)
worker_env = lora_runtime_environment(OFFLINE_ASSETS_ROOT, RUNTIME_DIR)
print("Locked caption experiment:", protocol_id)

In [ ]:
run_caption_worker("train", PAYLOAD_PATH, RUNTIME_DIR, worker_env)
training_summary = json.loads((OUTPUT_DIR / "training_complete.json").read_text(encoding="utf-8"))
display(training_summary)

In [ ]:
import pandas as pd
history = pd.read_csv(OUTPUT_DIR / "history.csv")
display(history)
comparison = pd.read_csv(OUTPUT_DIR / "holdout_before_after.csv").fillna("")
display(comparison[["reference", "base_short_caption", "adapted_short_caption", "human_preference"]].head(12))
print("Inspect the holdout images and fill human_preference/review_notes in holdout_before_after.csv.")
print("Lower token NLL is not proof of visual correctness, leakage absence, or improved M-TACS F1.")
if training_summary["best_holdout_token_nll"] >= training_summary["baseline_holdout_token_nll"]:
    print("Warning: adapted holdout NLL does not beat the frozen base; inspect data/overfitting before using it downstream.")

## 4. Export a New, Resumable `captions.csv`

Only after training completes: apply the fixed best adapter to all 5,640 images. The generation prompt uses pixels
and the same task token only. Dataset class names appear only in the **post-generation audit**, not caption generation.
The CSV uses **dtd_lexical_v2**: remove boilerplate, keep 5-8 words, reject direct class-term hits. CLIP maximum cosine
at 0.30 is **diagnostic only**; it never gates caption validity. This is not a certified semantic-leakage detector.
When `RUN_CLIP_AUDIT=False`, score fields remain empty with `semantic_audit_status=not_run`, not fabricated zeros.

Every successful batch is flushed; rerunning resumes missing image IDs. Model/adapter/policy/asset fingerprints
prevent mixing different captioners into a single CSV. Existing ablation captions/results are not overwritten.
Do not change decoding/policy based on test metrics. Adapter outputs are machine captions, not human ground truth.

In [ ]:
run_caption_worker("export", PAYLOAD_PATH, RUNTIME_DIR, worker_env)
NEW_CAPTIONS_CSV = OUTPUT_DIR / "exports" / "captions.csv"
export_summary = json.loads((OUTPUT_DIR / "exports" / "export_summary.json").read_text(encoding="utf-8"))
display(export_summary)
exported = pd.read_csv(NEW_CAPTIONS_CSV)
display(exported.groupby("split").agg(images=("image_id", "size"), valid_captions=("caption_valid", "sum")))
print("New captions CSV:", NEW_CAPTIONS_CSV)
print("Next: attach this CSV to a separate M-TACS run and set DTD_CAPTIONS_INPUT_PATH to its attached path.")
print("Keep the same DTD split, seed, candidate pool, 50 epochs and dtd_lexical_v2 policy as the old comparison.")
print("Use a new M-TACS output/cache directory. Do not resume old classifier checkpoints or reuse old text embeddings.")

## Interpretation and Sources

Use the before/after holdout table to check visual faithfulness and generic/repeated descriptions, not just NLL.
Then compare old vs new captions with the **same M-TACS configuration** and more than one seed when resources allow.
The existing ablation set must remain a frozen baseline. A captioner can become more fluent while classification worsens.

This notebook supplies a training pipeline and an empty review task; no GPU training or improvement claim is bundled.
Starting with 188 reviewed images is a small pilot, not a guaranteed adaptation recipe. An optional VLM teacher would
be a separate annotation step with image-only prompts and human acceptance; no teacher is silently invoked here.

Primary implementation references: [Florence fine-tuning example](https://huggingface.co/blog/finetune-florence2),
[Microsoft Florence model code](https://huggingface.co/microsoft/Florence-2-base/blob/main/modeling_florence2.py),
[PEFT 0.14 LoRA guide](https://huggingface.co/docs/peft/v0.14.0/en/quicktour).